# Mission 1: M1-XVEC-001 학습 코드와 실제 실행 기록

아래 코드는 실제 GPU 실험에서 사용한 원본입니다. 이 노트북에서 학습을 다시 실행한 것으로 표시하지 않습니다.
학습 기록은 완료된 lab 작업 JSON에서 가져왔으며, 코드 셀의 execution_count와 outputs는 비워 둡니다.
학습은 Training만 사용합니다. 공식 Validation 결과는 학습 기록과 구분해 마지막 셀에 보존합니다.
재학습 시 Training TL/TS zip, DCC_RUN_ROOT, LAB_WORK, LAB_OUT을 지정하고 로컬 dcc_data를 PYTHONPATH에 추가하세요.

원본 코드: scripts/lab/m1-xvec-001.py
원본 실행 기록: 02_M1-XVEC-001.json

```json
{
  "commit": "971d089244ca3721726295778dba2b5904883cf7",
  "worker": "colab-b",
  "gpu": "Tesla T4",
  "started_at": "2026-10-09T21:50:11+00:00",
  "finished_at": "2026-10-10T02:18:47+00:00",
  "elapsed_min": 73.6,
  "metrics": {
    "exp_id": "M1-XVEC-001",
    "primary_metric": {
      "name": "accuracy",
      "value": 0.9738013698630137,
      "split": "internal_holdout"
    },
    "metrics": {
      "heads": {
        "acoustic_52d": {
          "accuracy": 0.9501712328767123,
          "confusion_matrix_labels": [
            "F",
            "M"
          ],
          "confusion_matrix": [
            [
              2978,
              130
            ],
            [
              161,
              2571
            ]
          ],
          "recall_F": 0.9581724581724582,
          "recall_M": 0.9410688140556369,
          "n_samples": 5840,
          "n_F": 3108,
          "n_M": 2732
        },
        "xvector_512d": {
          "accuracy": 0.9707191780821918,
          "confusion_matrix_labels": [
            "F",
            "M"
          ],
          "confusion_matrix": [
            [
              3031,
              77
            ],
            [
              94,
              2638
            ]
          ],
          "recall_F": 0.9752252252252253,
          "recall_M": 0.9655929721815519,
          "n_samples": 5840,
          "n_F": 3108,
          "n_M": 2732
        },
        "combined_564d": {
          "accuracy": 0.9738013698630137,
          "confusion_matrix_labels": [
            "F",
            "M"
          ],
          "confusion_matrix": [
            [
              3038,
              70
            ],
            [
              83,
              2649
            ]
          ],
          "recall_F": 0.9774774774774775,
          "recall_M": 0.969619326500732,
          "n_samples": 5840,
          "n_F": 3108,
          "n_M": 2732
        }
      },
      "paired_mcnemar": {
        "exact_two_sided_p": 3.5201781942590924e-20,
        "baseline_wrong_combined_right": 187,
        "baseline_right_combined_wrong": 49
      },
      "accuracy_gain": 0.023630136986301342,
      "extraction_calls_per_second": 3.0494505636099354,
      "extraction_seconds": 9575.495451034,
      "caller_duration_seconds": {
        "min": 0.872,
        "p25": 22.81675,
        "median": 30.252499999999998,
        "p75": 41.264250000000004,
        "p95": 62.90615,
        "max": 129.228
      },
      "caller_duration_mean_seconds": 33.54156393835617,
      "padded_calls": 0,
      "truncated_calls": 24425
    },
    "seed": 42,
    "train_rows": 23360,
    "eval_rows": 5840,
    "official_val": null,
    "ckpt": "best.ckpt",
    "notes": "Fixed combined head; no official Validation. LR interrupted fit restarts that head."
  }
}
```


In [ ]:
"""M1-XVEC-001: fixed 52D/x-vector/combined LR, Training-only 80:20.

No fine-tuning. sklearn LBFGS has no resumable epoch/optimizer API: each
head is an atomic stage; interruption during fit restarts only that head.
Feature chunks and completed heads survive runtime loss in LAB_OUT.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
from pathlib import Path
import random
import shutil
import tempfile
import time
import unicodedata
import zipfile

EXP_ID = "M1-XVEC-001"
MODEL_ID = "microsoft/wavlm-base-plus-sv"
# Public immutable HF revision: https://huggingface.co/microsoft/wavlm-base-plus-sv/tree/1bfd64eca136543feb28c5ffaf05381c6af33121
REVISION = "1bfd64eca136543feb28c5ffaf05381c6af33121"
SEED = 42
CONFIG = dict(exp_id=EXP_ID, model=MODEL_ID, revision=REVISION, seed=SEED,
              chunk_size=2000, expected_calls=29200, sample_rate=16000,
              max_seconds=20, min_seconds=.5, holdout_ratio=.2, schema=1)


def atomic(path, writer):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    os.close(fd)
    try:
        writer(Path(name))
        with open(name, "rb") as stream:
            os.fsync(stream.fileno())
        os.replace(name, path)
    finally:
        Path(name).unlink(missing_ok=True)


def write_json(path, value):
    atomic(path, lambda p: p.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                                 allow_nan=False), encoding="utf-8"))


def prepare_model():
    from huggingface_hub import snapshot_download
    from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
    root = Path(os.environ["DCC_RUN_ROOT"]) / "models" / "wavlm-base-plus-sv"
    marker = root / "dcc_revision.json"
    identity = dict(repo_id=MODEL_ID, revision=REVISION)
    if marker.exists():
        if json.loads(marker.read_text()) != identity:
            raise ValueError(f"Model revision mismatch: {root}")
    else:
        snapshot_download(repo_id=MODEL_ID, revision=REVISION, local_dir=str(root),
                          allow_patterns=["config.json", "preprocessor_config.json",
                                          "pytorch_model.bin", "model.safetensors"])
    processor = Wav2Vec2FeatureExtractor.from_pretrained(root, local_files_only=True)
    model = WavLMForXVector.from_pretrained(root, local_files_only=True)
    if model.config.xvector_output_dim != 512:
        raise ValueError("Expected a 512-dimensional x-vector")
    write_json(marker, identity)
    return processor, model


def unpack(source, work):
    """Copy one specified archive; enumerate members only on local disk."""
    local = work / source.name
    if not local.exists():
        atomic(local, lambda p: shutil.copyfile(source, p))
    destination = work / source.stem
    marker = destination / ".complete"
    with zipfile.ZipFile(local) as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"Unsafe zip path: {member.filename}")
        if not marker.exists():
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
            atomic(marker, lambda p: p.write_text("complete"))
    return destination, [m.filename for m in members]


def prepare_rows(work):
    from dcc_data.colab_env import find_data_root
    from dcc_data.manifest import ManifestRow
    from dcc_data.mission1 import _read_gender
    source = find_data_root() / "Training"
    labels, label_members = unpack(source / "TL_서울_구급.zip", work)
    audio, audio_members = unpack(source / "TS_서울_구급.zip", work)
    def index(root, members, suffix):
        found = {}
        for name in members:
            path = Path(name)
            if path.suffix.lower() != suffix:
                continue
            key = unicodedata.normalize("NFC", path.stem).casefold()
            if key in found:
                raise ValueError(f"Duplicate call stem: {key}")
            found[key] = root / path
        return found
    jsons = index(labels, label_members, ".json")
    wavs = index(audio, audio_members, ".wav")
    if jsons.keys() != wavs.keys() or len(jsons) != CONFIG["expected_calls"]:
        raise ValueError(f"Expected 29,200 paired calls; JSON={len(jsons)}, WAV={len(wavs)}")
    rows, genders, calls = [], [], []
    for key in sorted(jsons):
        path = jsons[key]
        doc = json.loads(path.read_text(encoding="utf-8-sig"))
        gender = _read_gender(doc, path)
        if gender not in {"F", "M"}:
            raise ValueError(f"Invalid gender: {path}")
        calls.append(str(doc.get("call_id") or doc.get("_id") or key))
        genders.append(gender)
        rows.append(ManifestRow("Training", path.stem,
                                str(wavs[key].relative_to(work)), str(path.relative_to(work))))
    if len(set(calls)) != len(calls):
        raise ValueError("Multiple samples for the same call_id")
    return rows, genders, calls


def save_arrays(path, **arrays):
    import numpy as np
    def writer(p):
        with p.open("wb") as stream:
            np.savez(stream, **arrays)
    atomic(path, writer)


def extract(out, work, rows, processor, model, torch, np):
    import librosa
    from dcc_data.mission1 import extract_caller_segments, concatenate_caller_waveforms
    from dcc_data.mission2_features import acoustic_feature_vector
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    files = []
    for start in range(0, len(rows), CONFIG["chunk_size"]):
        stop = min(len(rows), start + CONFIG["chunk_size"])
        path = out / "features" / f"chunk_{start:05d}_{stop:05d}.npz"
        files.append(path)
        if path.exists():
            with np.load(path, allow_pickle=False) as chunk:
                if chunk["acoustic"].shape != (stop-start, 52) or chunk["xvector"].shape != (stop-start, 512):
                    raise ValueError(f"Invalid completed chunk: {path}")
            continue
        acoustic, xvectors, durations = [], [], []
        began = time.perf_counter()
        for row in rows[start:stop]:
            segments = extract_caller_segments(row, work)
            waveform = concatenate_caller_waveforms(segments)
            if waveform.ndim != 2 or waveform.shape[0] != 1:
                raise ValueError(f"Expected mono audio: {row.stem}")
            sr = segments[0].sample_rate
            acoustic.append(acoustic_feature_vector(waveform[0], sr))
            durations.append(waveform.shape[1] / sr)
            # Resample only the prefix needed for the x-vector, avoiding long-call work.
            prefix = waveform[0, :20 * sr]
            if sr != 16000:
                prefix = librosa.resample(prefix, orig_sr=sr, target_sr=16000)
            prefix = np.asarray(prefix[:320000], dtype=np.float32)
            prefix = np.pad(prefix, (0, max(0, 8000 - len(prefix))))
            inputs = processor(prefix, sampling_rate=16000, return_tensors="pt",
                               return_attention_mask=True)
            with torch.inference_mode():
                embedding = model(**{k: v.to(device) for k, v in inputs.items()}).embeddings
                embedding = torch.nn.functional.normalize(embedding, dim=-1)
            xvectors.append(embedding[0].cpu().numpy())
        if device == "cuda":
            torch.cuda.synchronize()
        a, x = np.asarray(acoustic), np.asarray(xvectors)
        if a.shape != (stop-start, 52) or x.shape != (stop-start, 512) or not np.isfinite(a).all() or not np.isfinite(x).all():
            raise ValueError("Invalid extracted features")
        save_arrays(path, acoustic=a, xvector=x, durations=np.asarray(durations),
                    elapsed=np.asarray(time.perf_counter()-began))
        print(f"Completed feature chunk {start}:{stop}", flush=True)
    chunks = [np.load(path, allow_pickle=False) for path in files]
    try:
        return (np.concatenate([c["acoustic"] for c in chunks]),
                np.concatenate([c["xvector"] for c in chunks]),
                np.concatenate([c["durations"] for c in chunks]),
                sum(float(c["elapsed"]) for c in chunks))
    finally:
        for chunk in chunks:
            chunk.close()


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=os.environ.get("LAB_OUT"))
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    if args.smoke:
        smoke()
        return
    # Fail before data copying/extraction on missing or incompatible local weights.
    processor, model = prepare_model()
    import numpy as np
    import torch
    import joblib
    from scipy.stats import binomtest
    from sklearn.model_selection import train_test_split
    from dcc_data.mission1_train import _build_lr_pipeline, compute_metrics, encode_gender
    if args.out is None:
        parser.error("--out or LAB_OUT is required")
    if os.environ.get("LAB_EXP_ID", EXP_ID) != EXP_ID:
        raise ValueError("LAB_EXP_ID mismatch")
    out = args.out
    work = Path(os.environ["LAB_WORK"]) / EXP_ID.lower()
    out.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if (out / "config.json").exists() and json.loads((out / "config.json").read_text()) != CONFIG:
        raise ValueError("Output configuration mismatch")
    write_json(out / "config.json", CONFIG)
    if (out / "metrics.json").exists() and (out / "best.ckpt").exists():
        print("Experiment already complete", flush=True)
        return
    rows, genders, calls = prepare_rows(work)
    identity = dict(calls=calls, genders=genders, paths=[(r.wav_path, r.json_path) for r in rows])
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    y = encode_gender(genders)
    train, holdout = train_test_split(np.arange(len(rows)), test_size=.2,
                                     random_state=SEED, stratify=y)
    split = dict(rows_sha256=digest, train=train.tolist(), holdout=holdout.tolist())
    if (out / "split.json").exists() and json.loads((out / "split.json").read_text()) != split:
        raise ValueError("Cached rows/split mismatch")
    write_json(out / "split.json", split)
    assert not ({calls[i] for i in train} & {calls[i] for i in holdout})
    a, x, durations, elapsed = extract(out, work, rows, processor, model, torch, np)
    del model
    scores, predictions = {}, {}
    for name, features in [("acoustic_52d", a), ("xvector_512d", x),
                           ("combined_564d", np.concatenate([a, x], axis=1))]:
        path = out / ("best.ckpt" if name == "combined_564d" else name + ".ckpt")
        if path.exists():
            pipeline = joblib.load(path)
        else:
            pipeline = _build_lr_pipeline(SEED)
            pipeline.fit(features[train], y[train])
            atomic(path, lambda p: joblib.dump(pipeline, p))
        predictions[name] = pipeline.predict(features[holdout])
        scores[name] = compute_metrics(y[holdout], predictions[name])
        write_json(out / (name + "_metrics.json"), scores[name])
    baseline = predictions["acoustic_52d"] == y[holdout]
    combined = predictions["combined_564d"] == y[holdout]
    improved = int(np.sum(~baseline & combined))
    regressed = int(np.sum(baseline & ~combined))
    pvalue = float(binomtest(improved, improved + regressed, .5).pvalue) if improved + regressed else 1.
    save_arrays(out / "holdout_predictions.npz", indices=holdout, targets=y[holdout], **predictions)
    quantiles = dict(zip(["min", "p25", "median", "p75", "p95", "max"],
                         np.quantile(durations, [0, .25, .5, .75, .95, 1]).tolist()))
    gain = scores["combined_564d"]["accuracy"] - scores["acoustic_52d"]["accuracy"]
    write_json(out / "metrics.json", dict(
        exp_id=EXP_ID, primary_metric=dict(name="accuracy", value=scores["combined_564d"]["accuracy"],
                                          split="internal_holdout"),
        metrics=dict(heads=scores, paired_mcnemar=dict(exact_two_sided_p=pvalue,
                     baseline_wrong_combined_right=improved, baseline_right_combined_wrong=regressed),
                     accuracy_gain= gain, extraction_calls_per_second=len(rows)/elapsed,
                     extraction_seconds=elapsed, caller_duration_seconds=quantiles,
                     caller_duration_mean_seconds=float(durations.mean()),
                     padded_calls=int(np.sum(durations < .5)), truncated_calls=int(np.sum(durations > 20))),
        seed=SEED, train_rows=len(train), eval_rows=len(holdout), official_val=None,
        ckpt="best.ckpt", notes="Fixed combined head; no official Validation. LR interrupted fit restarts that head."))


def smoke():
    """Dependency-free, no model download, no training, no official data access."""
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        source = root / "TL_서울_구급.zip"
        with zipfile.ZipFile(source, "w") as archive:
            archive.writestr("TL_서울_구급/example.json", '{"gender":"F"}')
        extracted, members = unpack(source, root / "work")
        assert json.loads((extracted / members[0]).read_text())["gender"] == "F"
        assert unpack(source, root / "work")[1] == members
        target = root / "atomic.json"
        write_json(target, CONFIG)
        assert json.loads(target.read_text()) == CONFIG
        write_json(target, {"resumed": True})
        assert json.loads(target.read_text()) == {"resumed": True}
        unsafe = root / "unsafe.zip"
        with zipfile.ZipFile(unsafe, "w") as archive:
            archive.writestr("../escape", "bad")
        try:
            unpack(unsafe, root / "work")
        except ValueError:
            pass
        else:
            raise AssertionError("Unsafe zip accepted")
    print("Smoke OK: atomic writes, archive staging/resume, path traversal guard; no training")




In [ ]:
# Run explicitly only when reproducing training on the configured GPU environment.
# import sys
# sys.argv = ['training', '--out', str(Path(os.environ['LAB_OUT']))]
# main()


## 제출 체크포인트의 공식 Validation 검증 기록

```json
{
  "exp_id": "M1-XVEC-001-OFFVAL",
  "primary_metric": {
    "name": "accuracy",
    "value": 0.9747252747252747,
    "split": "official_validation"
  },
  "identity": {
    "exp_id": "M1-XVEC-001",
    "split": "official_validation",
    "rows": 3640,
    "head_sha256": "4fffb8aa97c2f3083b05820ca45b84ecddbc60c5162e02961546f37d133dd9d5",
    "model_revision": {
      "repo_id": "microsoft/wavlm-base-plus-sv",
      "revision": "1bfd64eca136543feb28c5ffaf05381c6af33121"
    },
    "encoder_sha256": "94c3defe08248d81c7b2bd0a058ea9985269cefed13076434669c47fade41182",
    "input_sha256": "6ff3ab1731c9b8651c2e2ecf985b1f12107869fcc2690b456463f7723b2e1218",
    "max_seconds": 20,
    "min_seconds": 0.5
  },
  "metrics": {
    "accuracy": 0.9747252747252747,
    "confusion_matrix_labels": [
      "F",
      "M"
    ],
    "confusion_matrix": [
      [
        1919,
        40
      ],
      [
        52,
        1629
      ]
    ],
    "recall_F": 0.9795814190913732,
    "recall_M": 0.9690660321237359,
    "n_samples": 3640,
    "n_F": 1959,
    "n_M": 1681
  },
  "seed": 42,
  "eval_rows": 3640,
  "official_val": true,
  "elapsed_seconds": 1212.6283473580006,
  "threads": 2,
  "fitting_performed": false,
  "inference_device": "cuda",
  "comparison": {
    "baseline_accuracy": 0.951098901098901,
    "baseline_correct": 3462,
    "candidate_correct": 3548,
    "delta_correct": 86,
    "delta_percentage_points": 2.3626373626373653,
    "minimum_gain_percentage_points": 0.5,
    "decision": "adopt_xvector_candidate"
  },
  "verification": {
    "lab_status": "done",
    "gpu": "Tesla T4",
    "official_csv_rows": 3640,
    "unique_filenames": true,
    "genders_valid": true,
    "local_input_head_encoder_identity_matches": true,
    "cpu_duplicate_cancelled_after_gpu_verification": true,
    "partial_cpu_cache_preserved": true,
    "csv_independently_rescored_locally": true
  },
  "execution": {
    "code_commit": "3db888ec38b1694893b42176a45afc64a8aa4541",
    "worker": "colab-a",
    "finished_at_kst": "2026-10-10T18:11:58+09:00",
    "lab_elapsed_minutes": 23.2,
    "metric_source": "outputs/model_selection/m1_xvec001_gpu/metrics.json"
  }
}
```
